In [4]:
import os
import sys
sys.path.append('../modules')

from crd_processor import CRDProcessor
from orbit_validator import OrbitValidator
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

# Widgets for interactive control
from ipywidgets import interact, FloatSlider

plt.style.use('ggplot')
os.makedirs('../processed', exist_ok=True)

IndentationError: expected an indented block (crd_processor.py, line 14)

In [3]:
processor = CRDProcessor()
df = processor.process_directory('../envisat')

# Calculate anomalies
df['anomaly'] = (np.abs(df['range'] - df['range'].median()) > 
                 3*df['range'].std())
df.to_csv('../processed/envisat_processed.csv', index=False)

NameError: name 'CRDProcessor' is not defined

In [ ]:
validator = OrbitValidator()
tle = validator.get_tle(27386)  # Envisat NORAD ID

# Run validations
violations = validator.validate_physics(df)
tle_comparison = validator.compare_with_tle(df, tle)
plot_path = validator.generate_plots(df, '../processed')

In [ ]:
@interact(
    sigma_threshold=FloatSlider(value=3, min=1, max=5, step=0.5),
    min_snr=FloatSlider(value=20, min=5, max=50)
)
def analyze(sigma_threshold, min_snr):
    filtered = df[df['snr'] > min_snr].copy()
    filtered['anomaly'] = (np.abs(filtered['range'] - filtered['range'].median()) > 
                          sigma_threshold*filtered['range'].std())
    
    # Display results
    display(Markdown(f"**Anomalies detected:** {filtered['anomaly'].sum()}"))
    
    fig, ax = plt.subplots(figsize=(10,5))
    ax.scatter(filtered['timestamp'], filtered['range'], 
               c=filtered['anomaly'], cmap='coolwarm')
    plt.show()

In [ ]:
# Generate report
report = f"""
# Envisat Monitoring Report
## Validation Summary
- Physical Violations: {len(violations)}
- TLE Mean Error: {tle_comparison['mean_error']:.6f} ls
![Validation Plots]({plot_path})
"""

display(Markdown(report))

# Save report
with open('../processed/latest_report.md', 'w') as f:
    f.write(report)

# AWS Integration
if 'AWS_ACCESS_KEY_ID' in os.environ:
    import boto3
    s3 = boto3.client('s3')
    s3.upload_file('../processed/latest_report.md', 
                  'your-bucket-name', 
                  'reports/envisat_latest.md')

In [ ]:
# Dashboard with refresh button
from IPython.display import clear_output
from ipywidgets import Button

def update_dashboard(b):
    clear_output()
    display(HTML("<h2>Live Envisat Monitoring</h2>"))
    display(HTML(f"<p>Last Updated: {pd.Timestamp.now()}</p>"))
    
    # Re-run validations
    current_violations = validator.validate_physics(df)
    display(HTML("<h3>Active Alerts</h3>"))
    for v in current_violations:
        display(HTML(f"<div style='color:red'>⚠️ {v}</div>"))
    
    # Show latest plot
    display(HTML(f"<img src='{plot_path}' width=800>"))

refresh = Button(description="Refresh Dashboard")
refresh.on_click(update_dashboard)
display(refresh)
update_dashboard(None)  # Initial load

In [1]:
# Schedule daily runs
import schedule
import time

def daily_task():
    print("Running scheduled analysis...")
    %run -i notebooks/Envisat_Monitoring.ipynb

schedule.every().day.at("09:00").do(daily_task)

# Keep the notebook alive
while True:
    schedule.run_pending()
    time.sleep(60)

ModuleNotFoundError: No module named 'schedule'

In [ ]:
from sklearn.ensemble import IsolationForest

clf = IsolationForest(contamination=0.01)
df['ml_anomaly'] = clf.fit_predict(df[['timestamp', 'range']].values) == -1